Experiment 1

In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import lightgbm as lgb
import xgboost as xgb

In [37]:
df = pd.read_csv('../data/raw/diabetes.csv')
X = df.drop('Outcome', axis=1)
y = df['Outcome']

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

Train shape: (614, 8), Test shape: (154, 8)


In [42]:
def evaluate_model(name, y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    print(f"\n{name}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    
    if y_prob is not None:
        auc = roc_auc_score(y_true, y_prob)
        print(f"AUC-ROC  : {auc:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

In [43]:
lr = LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

evaluate_model("Logistic Regression (raw)", y_test, lr_pred, lr_prob)


Logistic Regression (raw)
Accuracy : 0.7273
Precision: 0.6429
Recall   : 0.5000
F1-score : 0.5625
AUC-ROC  : 0.8287

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.85      0.80       100
           1       0.64      0.50      0.56        54

    accuracy                           0.73       154
   macro avg       0.70      0.68      0.68       154
weighted avg       0.72      0.73      0.72       154



In [44]:
lgb_model = lgb.LGBMClassifier(
    random_state=42,
    verbosity=-1,
    n_estimators=200,
)

lgb_model.fit(X_train, y_train)

lgb_pred = lgb_model.predict(X_test)
lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
evaluate_model("LightGBM (raw)", y_test, lgb_pred, lgb_prob)


LightGBM (raw)
Accuracy : 0.7143
Precision: 0.6042
Recall   : 0.5370
F1-score : 0.5686
AUC-ROC  : 0.8056

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.81      0.79       100
           1       0.60      0.54      0.57        54

    accuracy                           0.71       154
   macro avg       0.68      0.67      0.68       154
weighted avg       0.71      0.71      0.71       154



In [45]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_estimators=200,
    verbosity=0
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

evaluate_model("XGBoost (Baseline)", y_test, xgb_pred, xgb_prob)


XGBoost (Baseline)
Accuracy : 0.7532
Precision: 0.6600
Recall   : 0.6111
F1-score : 0.6346
AUC-ROC  : 0.8059

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.83      0.81       100
           1       0.66      0.61      0.63        54

    accuracy                           0.75       154
   macro avg       0.73      0.72      0.72       154
weighted avg       0.75      0.75      0.75       154



In [47]:
print("RAW MODEL COMPARISON")
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
print("Class Distribution:", y.value_counts().to_dict())

models = {
    "Logistic Regression": lr,
    "LightGBM": lgb_model,
    "XGBoost": xgb_model
}

for name, model in models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob)
    f1 = f1_score(y_test, pred)
    print(f"\n{name:20} AUC: {auc:.4f} F1: {f1:.4f}")

RAW MODEL COMPARISON
Train size: 614, Test size: 154
Class Distribution: {0: 500, 1: 268}

Logistic Regression  AUC: 0.8287 F1: 0.5625

LightGBM             AUC: 0.8056 F1: 0.5686

XGBoost              AUC: 0.8059 F1: 0.6346
